In [1]:
import os
import math
import random
import numpy as np
import pandas 
import fiona
import logging

import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPoint, MultiPolygon, LineString
from shapely.ops import unary_union, voronoi_diagram, nearest_points
from shapely import affinity

from shapely.strtree import STRtree

In [2]:
def find_intersecting_assets(flood_polygon, layer):
    # Ensure both layer and flood_polygon are in the same CRS
    layer = layer.to_crs(flood_polygon.crs)
    
    # Perform the spatial join where the geometries intersect
    intersecting_assets = gpd.sjoin(layer, flood_polygon, how="inner", predicate="intersects")
    
    return intersecting_assets


In [3]:
RP = ['100']
RCP = ['baseline2010', '262050', '262100', '452030', '452050', '452070', '452100', '852030', '852050', '852070', '852100']


#------------------ FILE PATHS -----------------------
networks_csv = "data/network_layers.csv"
data_path = "/Users/adam/PROJECTS/JSRAT/JSRAT-GITS/jamaica-infrastructure/processed_data/"
networks = pandas.read_csv(networks_csv)

flood_areas = "outputs/Jamaica_coastal_protection_areas.gpkg"
flood_layer = "flood_protection_area_rcp_baseline2010_rp100"

coastal_buffer = "outputs/coastal_protection_processing_layers.gpkg"
coastal_buffer_layer = "jamaica_convex"
#------------------------------------------------------

In [4]:
def find_intersecting_assets(flood_polygon, layer):
    # Ensure both layer and flood_polygon are in the same CRS
    layer = layer.to_crs(flood_polygon.crs)
    
    # Perform the spatial join where the geometries intersect
    intersecting_assets = gpd.sjoin(layer, flood_polygon, how="inner", predicate="intersects")
    
    return intersecting_assets

In [5]:
def join_flood_areas(flood_polygons):
    # Merge all polygons into a single polygon
    merged_polygon = unary_union(flood_polygons.geometry)
    return merged_polygon

all_flood_areas = []

for rcp in RCP:
    for rp in RP:
        flood_polygon_layer = f"flood_protection_area_rcp_{rcp}_rp{rp}"
        flood_polygons = gpd.read_file(flood_areas, layer=flood_polygon_layer)
        merged_flood_polygon = join_flood_areas(flood_polygons)
        all_flood_areas.append(merged_flood_polygon)

# Create a union of all merged flood areas
final_flood_area = unary_union(all_flood_areas)

# Save the final merged flood polygon
union_flood_area_gdf = gpd.GeoDataFrame(geometry=[final_flood_area], crs=flood_polygons.crs)
union_flood_area_gdf.to_file("outputs/full_coastal_area.gpkg", driver="GPKG")


In [6]:
# flood_polygon = gpd.read_file("outputs/full_coastal_area.gpkg")
flood_polygon = union_flood_area_gdf

flood = flood_polygon.loc[flood_polygon.index[0]]
flood_polygon = gpd.GeoDataFrame([flood], geometry='geometry', crs = flood_polygon.crs)

for index, n in networks.iterrows():
    fname = os.path.join(data_path, n['path'])
    id_col = n['asset_id_column']
    layer_type = n['gpkg_layer']
    ref = n['ref']
    
    assets = gpd.read_file(fname, layer=layer_type)
    if assets.empty:
        continue
    asset_intersections = find_intersecting_assets(flood_polygon, assets)
    asset_intersections.to_file("outputs/Filtered_Assets.gpkg", layer=ref, driver="GPKG")
    print ("Completed for", ref, "assets")

# flood_polygon.to_file("Intersection_polygon_test.gpkg", driver="GPKG")

Completed for transport_rail_nodes assets
Completed for transport_rail_edges assets
Completed for transport_air assets
Completed for transport_port assets
Completed for water_irrigation_nodes assets
Completed for water_irrigation_edges assets
Completed for water_potable_nodes assets
Completed for water_wastewater_nodes assets
Completed for water_pipelines assets
Completed for transport_roads_nodes assets
Completed for transport_roads_edges assets
Completed for energy_nodes assets
Completed for energy_edges assets
Completed for buildings assets


In [22]:
def Find_Assets_per_flood_area(flood_polygons, filtered_assets_path, rcp, rp):
    for index, flood_feature in flood_polygons.iterrows():
        print(f"   --   Processing Flood Polygon {index}   --   ")
        for _, n in networks.iterrows():
            fname = os.path.join(data_path, n['path'])
            id_col = n['asset_id_column']
            layer_type = n['gpkg_layer']
            ref = n['ref']

            assets = gpd.read_file(filtered_assets_path, layer=ref)
            assets = assets.drop(columns=["index_right"], errors="ignore")  # Avoid KeyError

            if assets.empty:
                # print(f"Skipping {ref} as no assets were found.")
                continue

            # Find intersecting assets
            asset_intersections = find_intersecting_assets(
                gpd.GeoDataFrame([flood_feature], geometry='geometry', crs=flood_polygons.crs), assets
            )

            # print(f"Expected ID Column: {id_col}")
            # print("Asset Intersections Columns:", asset_intersections.columns)

            if id_col in asset_intersections.columns:
                asset_ids = asset_intersections[id_col].tolist()
            else:
                # print(f"Warning: Column '{id_col}' not found in asset_intersections.")
                asset_ids = []

            # Store results in flood_polygons
            flood_polygons.at[index, f"{ref}__{id_col}"] = str(asset_ids)

            print(f"       -   Completed asset: {ref}")
        # break

    flood_polygons.to_file(
        "outputs/flood_Protection_area_with_assets.gpkg",
        layer=f"flood_asset_area_rcp_{rcp}_rp_{rp}",
        driver="GPKG"
    )


In [23]:
RCP = ['baseline2010', '262100', '452030', '452050', '452070', '452100', '852030', '852050', '852070', '852100']
# RCP = ['262050']
for rcp in RCP:
    for rp in RP:
        print (f"Current Layer: flood_protection_area_rcp_{rcp}_rp{rp}")
        flood_polygon_layer = f"flood_protection_area_rcp_{rcp}_rp{rp}"
        flood_polygons = gpd.read_file(flood_areas, layer=flood_polygon_layer)

        filtered_assets_path = "outputs/Filtered_Assets.gpkg"
        Find_Assets_per_flood_area(flood_polygons, filtered_assets_path, rcp, rp)
        

Current Layer: flood_protection_area_rcp_baseline2010_rp100
   --   Processing Flood Polygon 0   --   
       -   Completed asset: transport_rail_nodes
       -   Completed asset: transport_rail_edges
       -   Completed asset: transport_air
       -   Completed asset: transport_port
       -   Completed asset: water_irrigation_nodes
       -   Completed asset: water_irrigation_edges
       -   Completed asset: water_potable_nodes
       -   Completed asset: water_wastewater_nodes
       -   Completed asset: water_pipelines
       -   Completed asset: transport_roads_nodes
       -   Completed asset: transport_roads_edges
       -   Completed asset: energy_nodes
       -   Completed asset: energy_edges
       -   Completed asset: buildings
   --   Processing Flood Polygon 1   --   
       -   Completed asset: transport_rail_nodes
       -   Completed asset: transport_rail_edges
       -   Completed asset: transport_air
       -   Completed asset: transport_port
       -   Completed as